# 03 - Inference with the packaged model
Loads `artifacts/model/model.joblib` (preprocessing + classifier in one pipeline) and verifies that predictions are reproducible after reloading.

In [ ]:
# Make the repository's src/ importable whether or not `pip install -e .` was run
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)
import platform, socket, datetime
print("Run on host:", socket.gethostname(), "| Python", platform.python_version(),
      "| started", datetime.datetime.now().isoformat(timespec="seconds"))

In [ ]:
from heartrisk.model_io import load_model
import json
model = load_model()
print(json.dumps({k: model.metadata[k] for k in ['model_family','model_version','best_params','test_metrics']}, indent=2))

In [ ]:
sample = json.load(open('samples/sample_request.json'))
model.predict_records([sample])

Missing `ca` / `thal` are allowed (imputed exactly as during training):

In [ ]:
model.predict_records([{k: v for k, v in sample.items() if k not in ('ca', 'thal')}])

### Round-trip check on the cleaned dataset

In [ ]:
import joblib, numpy as np
from heartrisk.data import load_clean, split_xy
X, y = split_xy(load_clean())
a = joblib.load('artifacts/model/model.joblib').predict_proba(X)[:, 1]
b = np.array([r['probability_disease'] for r in model.predict_records(X.to_dict('records'))])
print('max |difference| =', float(np.abs(a - b).max()))
assert np.allclose(a, b, atol=1e-4)